# Jaynes–Cummings cavity dynamics

Cleaned and organized for reproducible execution from the repository root.


In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
from qutip import (about, basis, destroy, mesolve, ptrace, qeye,
                   tensor, wigner, anim_wigner)
# set a parameter to see animations in line
from matplotlib import rc
rc('animation', html='jshtml')

%matplotlib inline

In [ ]:
## Problem parameters
wc = 1.0 * 2 * np.pi  # cavity frequency
wa = 1.0 * 2 * np.pi  # atom frequency
g = 0.05 * 2 * np.pi  # coupling strength
kappa = 0.005  # cavity dissipation rate
gamma = 0.05  # atom dissipation rate
N = 15  # number of cavity fock states
n_th_a = 0.0  # avg number of thermal bath excitation
use_rwa = True

tlist = np.linspace(0, 25, 101)

In [ ]:
## Setup the operators, the Hamiltonian and initial state
# intial state
psi0 = tensor(basis(N, 0), basis(2, 1))  # start with an excited atom

# operators
a = tensor(destroy(N), qeye(2))
sm = tensor(qeye(N), destroy(2))

# Hamiltonian
if use_rwa:
    H = wc * a.dag() * a + wa * sm.dag() * sm + \
        g * (a.dag() * sm + a * sm.dag())
else:
    H = wc * a.dag() * a + wa * sm.dag() * sm + \
        g * (a.dag() + a) * (sm + sm.dag())

In [ ]:
## Create a list of collapse operators that describe the dissipation
c_ops = []

# cavity relaxation
rate = kappa * (1 + n_th_a)
if rate > 0.0:
    c_ops.append(np.sqrt(rate) * a)

# cavity excitation, if temperature > 0
rate = kappa * n_th_a
if rate > 0.0:
    c_ops.append(np.sqrt(rate) * a.dag())

# qubit relaxation
rate = gamma
if rate > 0.0:
    c_ops.append(np.sqrt(rate) * sm)

In [ ]:
## Evolve the system
output = mesolve(H, psi0, tlist, c_ops, [a.dag() * a, sm.dag() * sm])

In [ ]:
## Visualize the results
n_c = output.expect[0]
n_a = output.expect[1]

fig, axes = plt.subplots(1, 1, figsize=(10, 6))

axes.plot(tlist, n_c, label="Cavity")
axes.plot(tlist, n_a, label="Atom excited state")
axes.legend(loc=0)
axes.set_xlabel("Time")
axes.set_ylabel("Occupation probability")
axes.set_title("Vacuum Rabi oscillations")

$Cavity wigner function$

In [ ]:
output = mesolve(H, psi0, tlist, c_ops, [])

In [ ]:
output
type(output.states)
len(output.states)
# indexing the list with -1 results in the last element in the list
output.states[-1]

In [ ]:
# find the indices of the density matrices for the times we are interested in
t_idx = np.where([tlist == t for t in [0.0, 5.0, 15.0, 25.0]])[1]
tlist[t_idx]

In [ ]:
# get a list density matrices
rho_list = [output.states[i] for i in t_idx]

In [ ]:
# loop over the list of density matrices

xvec = np.linspace(-3, 3, 200)

fig, axes = plt.subplots(1, len(rho_list), sharex=True,
                         figsize=(3 * len(rho_list), 3))

for idx, rho in enumerate(rho_list):

    # trace out the atom from the density matrix, to obtain
    # the reduced density matrix for the cavity
    rho_cavity = ptrace(rho, 0)

    # calculate its wigner function
    W = wigner(rho_cavity, xvec, xvec)

    # plot its wigner function
    axes[idx].contourf(
        xvec,
        xvec,
        W,
        100,
        norm=mpl.colors.Normalize(-0.25, 0.25),
        cmap=plt.get_cmap("RdBu"),
    )

    axes[idx].set_title(r"$t = %.1f$" % tlist[t_idx][idx], fontsize=16)

$解释生成图像$

t=0：初态通常是一个单光子费米子数态（Fock $n=1$），其 Wigner 函数在相空间原点处出现明显的负值（深蓝），这是量子非经典性的典型特征。

t=5：在 Jaynes–Cummings 相互作用下，腔场态经历了一次能量交换，Wigner 函数中心转为正值（红），并在周围出现负值环带，表明态变成了类似“猫态”或叠加态.

t=15：进一步演化后，态又回到类似相干态的高斯分布（主要正值），负值区域减弱。

t=25：又一次能量往返，Wigner 函数呈现环状高斯（带有小幅负值或接近零），对应于不同的费米子数分布或混合态.

展示了腔场在原子–腔耦合下的能量往复（Rabi 振荡）和随之产生的非经典相空间结构（猫态、负值区、相干态等）的周期性演化


In [ ]:
fig = plt.figure(figsize=(6, 6))
ax = fig.add_subplot(111, projection='3d')
ax.view_init(elev=0)

rho_cavity = list()

xvec = np.linspace(-3, 3, 150)

for idx, rho in enumerate(output.states):
    rho_cavity.append(ptrace(rho, 0))

fig, ani = anim_wigner(rho_cavity, xvec, xvec, projection='3d',
                       colorbar=True, fig=fig, ax=ax)

# close an auto-generated plot and animation
plt.close()
ani